# Comprehensive Machine Learning Model Comparison & Analysis

Welcome to this end-to-end masterclass notebook. The objective of this notebook is not just to write code, but to provide **deep theoretical insights, mathematical intuition, and rigorous comparative analysis** of various machine learning models.

We will use the **Breast Cancer Wisconsin (Diagnostic) Dataset** to predict whether a tumor is malignant or benign.

### Table of Contents
1. [Data Loading & Exploratory Data Analysis (EDA)](#1.-Data-Loading-&-Exploratory-Data-Analysis-(EDA))
2. [Data Preprocessing](#2.-Data-Preprocessing)
3. [Model 1: Logistic Regression](#3.-Model-1:-Logistic-Regression)
4. [Model 2: Support Vector Machines (SVM)](#4.-Model-2:-Support-Vector-Machines-(SVM))
5. [Model 3: Random Forest](#5.-Model-3:-Random-Forest)
6. [Comparative Analysis](#6.-Comparative-Analysis)
\n

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Configure plot aesthetics
sns.set_theme(style="whitegrid")
import warnings
warnings.filterwarnings('ignore')
\n

## 1. Data Loading & Exploratory Data Analysis (EDA)
Understanding the data is the most critical step. We load the dataset and look at the class distribution. Imbalanced datasets require special metrics (like F1-score) rather than just accuracy.\n

In [ ]:
# Load dataset
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

print(f"Dataset shape: {X.shape}")

# Visualizing target distribution
plt.figure(figsize=(6, 4))
sns.countplot(x=y, palette='viridis')
plt.title('Target Distribution (0 = Malignant, 1 = Benign)')
plt.show()
\n

## 2. Data Preprocessing
Machine learning models, especially distance-based ones like SVM and KNN, or optimization-based ones like Logistic Regression, require features to be on the same scale.

**Mathematical Note on Scaling:**
Standardization transforms the data to have a mean of 0 and a standard deviation of 1.
`z = (x - mean) / standard_deviation`\n

In [ ]:
# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Dictionary to store results for later comparison
results = {}

def evaluate_model(name, y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    results[name] = {'Accuracy': acc, 'Precision': prec, 'Recall': rec, 'F1 Score': f1}
    print(f"--- {name} ---")
    print(f"Accuracy : {acc:.4f}")
    print(f"Precision: {prec:.4f}")
    print(f"Recall   : {rec:.4f}")
    print(f"F1 Score : {f1:.4f}\n")
\n

## 3. Model 1: Logistic Regression

### Theory & Intuition
Logistic Regression is a linear model used for binary classification. Despite its name, it is a classification algorithm.

It calculates the weighted sum of inputs `z = w1*x1 + w2*x2 + ... + b`.
Instead of outputting `z` directly (like Linear Regression), it passes `z` through the **Sigmoid Activation Function**:
`sigma(z) = 1 / (1 + e^(-z))`

This function maps any real number into a probability between 0 and 1. If the probability is > 0.5, the model predicts class 1; otherwise, class 0.

**Pros**: Highly interpretable, fast to train, provides calibrated probabilities.
**Cons**: Assumes a linear decision boundary between classes.\n

In [ ]:
from sklearn.linear_model import LogisticRegression

log_reg = LogisticRegression(random_state=42)
log_reg.fit(X_train_scaled, y_train)

y_pred_log = log_reg.predict(X_test_scaled)
evaluate_model('Logistic Regression', y_test, y_pred_log)
\n

## 4. Model 2: Support Vector Machines (SVM)

### Theory & Intuition
The objective of a Support Vector Machine is to find a hyperplane in an N-dimensional space that distinctly classifies the data points while maximizing the margin.

The **Margin** is the distance between the hyperplane and the nearest data point from either class. These nearest points are called **Support Vectors**.

When data is not linearly separable, SVM uses the **Kernel Trick** (e.g., RBF kernel) to implicitly map inputs into high-dimensional feature spaces where a linear separator can be found.

**Pros**: Highly effective in high dimensional spaces.
**Cons**: Does not directly provide probability estimates; training can be slow on very large datasets.\n

In [ ]:
from sklearn.svm import SVC

svm_model = SVC(kernel='rbf', random_state=42)
svm_model.fit(X_train_scaled, y_train)

y_pred_svm = svm_model.predict(X_test_scaled)
evaluate_model('SVM (RBF Kernel)', y_test, y_pred_svm)
\n

## 5. Model 3: Random Forest

### Theory & Intuition
Random Forest is an **Ensemble Learning** method. Instead of relying on one single Decision Tree, it builds a "forest" of multiple decision trees.

**How it prevents overfitting (Bagging):**
1. **Bootstrapping**: Each tree is trained on a random subset of the data (with replacement).
2. **Feature Randomness**: At each split in the tree, only a random subset of features is considered.

The final prediction is made by taking a majority vote among all the trees in the forest.

**Pros**: Extremely robust to overfitting, requires almost no data preprocessing (scaling is not strictly necessary), handles non-linear relationships well.
**Cons**: Can be a "black box" (hard to interpret exactly why a prediction was made compared to a single tree).\n

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Note: Random Forests don't strictly require scaled data, but we use it here for consistency
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train_scaled, y_train)

y_pred_rf = rf_model.predict(X_test_scaled)
evaluate_model('Random Forest', y_test, y_pred_rf)
\n

## 6. Comparative Analysis

Now that we have trained our models, we must compare them rigorously. We look at four main metrics:
- **Accuracy**: Overall correctness.
- **Precision**: Out of all predicted positives, how many were actually positive? (Crucial when false positives are costly).
- **Recall**: Out of all actual positives, how many did we find? (Crucial in medical diagnosis where false negatives are deadly).
- **F1-Score**: The harmonic mean of Precision and Recall.

Let's visualize the performance.\n

In [ ]:
# Convert results dictionary to DataFrame for easy plotting
results_df = pd.DataFrame(results).T

# Plotting
results_df.plot(kind='bar', figsize=(10, 6), colormap='viridis')
plt.title('Model Performance Comparison')
plt.ylabel('Score')
plt.ylim(0.9, 1.0) # Zoom in to see the differences
plt.xticks(rotation=0)
plt.legend(loc='lower right')
plt.show()

display(results_df)
\n

### Final Conclusion
In medical datasets like this Breast Cancer diagnostic set, **Recall** is often the most critical metric. Missing a malignant tumor (False Negative) is much worse than a false alarm (False Positive). 

By analyzing the bar chart and the table above, we can determine which model provides the best balance of Recall and overall Accuracy, making it the ideal candidate for clinical deployment.\n